# Combining the extreme graphs with their source batch

The GA in `extreme_graphs.ipynb` evolves topologies *against* ML models fitted on one
source batch, so the extreme graphs and that source batch only mean something side by
side. This notebook produces a single batch directory holding both, ready for
`experiment_analysis.ipynb`.

Two steps, and they are independent:

1. **Simulate the extreme graphs** as their own ordinary batch (section 1). Run once,
   then leave it commented out.
2. **Combine** that batch with the source batch into a new directory (section 2).

The combine does *not* re-simulate and does *not* copy the raw data. It unions the two
parents' `graph_props.csv` and `graph_statistics.csv`, and **symlinks** their raw parquet
shards into `tmp/results/` under a single renumbered sequence. So the combined batch costs
a few MB, and every analysis path -- including the raw-data ones like the violin plot and
the speed report -- works on it unchanged.

Why concatenating the rollups is exact and not an approximation: `graph_statistics.csv`
is grouped by `(wl_hash, r)` and every statistic in it decomposes into additive partials,
so the union of the parents' rollups equals re-aggregating the union of their raw rows --
*provided* no `(wl_hash, r)` key appears in both parents. `combine_batches` verifies that
and refuses rather than silently discarding one parent's runs.

In [ ]:
%load_ext autoreload
%autoreload 2
%cd /home/labs/pilpel/matanyaw/moran-process

import sys

sys.path.insert(0, "src")

import joblib
from pathlib import Path

from moran_process.core.graph_zoo import GraphZoo

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
SIM_DATA = PROJECT_ROOT / "simulation_data"

# The batch the ML models were fitted on (random + respiratory graphs).
SOURCE_BATCH = "2026_07_15-respiratory-vs-random-100K-reps-2"
# The batch holding ONLY the GA-evolved extreme graphs (built in section 1).
EXTREME_BATCH = "2026_07_20-extreme_ocmbined_100K-2"
# The combined batch this notebook produces (section 2). Feed this to experiment_analysis.
COMBINED_BATCH = "2026_07_21-combined-respiratory-random-extreme"

## 1. Simulate the extreme graphs (run once)

The GA saves its winners as a nested list (one list per optimisation target), so they are
flattened into a single zoo. `GraphZoo` deduplicates by WL hash on the way in, which is why
the submitted count and the registered count can differ: 80 graphs were submitted and 71
were distinct, because one target's 10 winners were all isomorphic to each other.

**Already run for `EXTREME_BATCH`.** Uncomment only to build a new extreme batch.

In [ ]:
# --- Build the extreme zoo from the GA output ---------------------------------------
# extreme_graph_path = SIM_DATA / SOURCE_BATCH / "extreme_graph_zoo" / "extreme_graphs.joblib"
# extreme_graph_lists = joblib.load(extreme_graph_path)
#
# extreme_zoo = GraphZoo()
# for target_winners in extreme_graph_lists:
#     for graph in target_winners:
#         extreme_zoo.add(graph)
#
# (SIM_DATA / EXTREME_BATCH).mkdir(parents=True, exist_ok=True)
# extreme_zoo.save(str(SIM_DATA / EXTREME_BATCH / "extreme_zoo.pkl"))
# print(f"{len(extreme_zoo)} extreme graphs saved")

In [ ]:
# --- Submit the extreme batch to LSF -------------------------------------------------
# The r values and n_repeats MUST match the source batch, otherwise the two rollups are
# not comparable and the combined batch is meaningless.
#
# from moran_process.pipeline.process_lab import ProcessLab
#
# R_VALUES = [1, 1.1, 1.2, 1.3, 1.5, 2]
# N_REPEATS = 100_000
# N_JOBS = 1000
# SEED = 42
# ENGINE = "cpp"
#
# lab = ProcessLab()
# lab.submit_jobs(
#     zoo_path=str(SIM_DATA / EXTREME_BATCH / "extreme_zoo.pkl"),
#     r_values=R_VALUES,
#     n_repeats=N_REPEATS,
#     n_requested_jobs=N_JOBS,
#     n_graphs=len(extreme_zoo),
#     queue="gsla-cpu",
#     memory="1GB",
#     batch_dir=str(SIM_DATA / EXTREME_BATCH),
#     batch_name=EXTREME_BATCH,
#     graph_types=sorted({g.category for g in extreme_zoo}),
#     node_sizes=sorted({g.n_nodes for g in extreme_zoo}),
#     batch_seed=SEED,
#     engine=ENGINE,
#     description=f"Extreme graphs evolved against the ML models of {SOURCE_BATCH}",
#     notes="",
# )

## 2. Combine

Both parents must be finished, i.e. each already has `graph_statistics.csv` (the dependent
aggregation job writes it automatically once the LSF array completes). If one is missing,
build it first:

```bash
bsub -q short -R "rusage[mem=16384]" \
  uv run python -m moran_process.pipeline.aggregate_batch --batch-dir simulation_data/<batch>
```

In [ ]:
from moran_process.pipeline.combine_batches import combine_batches

combine_batches(
    source_dirs=[SIM_DATA / SOURCE_BATCH, SIM_DATA / EXTREME_BATCH],
    dest_dir=SIM_DATA / COMBINED_BATCH,
    description=(
        f"Union of {SOURCE_BATCH} (random + respiratory) and {EXTREME_BATCH} "
        "(GA-evolved extreme graphs designed against its residual ML models)."
    ),
    notes=(
        "graph_statistics.csv is the concatenation of both parents' rollups (exact: zero "
        "(wl_hash, r) overlap). Raw shards are symlinks into the parents, not copies."
    ),
    link_raw=True,        # symlink both parents' raw shards; costs no disk
    overwrite=True,       # the combine is cheap and idempotent, so just rebuild it
    submit_post_jobs=True,  # bsub the QC report + violin cache (see section 3)
)

## 3. Post-batch jobs (QC report + violin cache)

`combine_batches` submits these automatically, and so does `ProcessLab.submit_jobs` for a
simulated batch (there chained behind the aggregation with `bsub -w`). Use the cell below
only to (re)run them on a batch that already exists.

**QC report** (`pipeline/batch_report.py`) writes `report/qc_report.{json,txt}`: shard
completeness, join coverage, r coverage, NaN, and runs-per-cell. It reads only the CSVs and
the parquet *footers*, so it is seconds of work on any batch size.

**Violin cache** (`pipeline/cache_violin_data.py`) writes
`cache/fixation_steps_r<r>_n<cap>.parquet`. This is the one step that still scans the raw
shards, so it must not run on the login node. Afterwards, pass `cache_dir=BATCH_DIR` to
`plot_steps_violin` / `plot_steps_pvalue_matrix` and they read a few MB instead of 7.2e9
rows.

In [ ]:
# Re-run the post-batch jobs on a batch that already exists (both bsub immediately).
# combine_batches and ProcessLab.submit_jobs already do this for new batches.

from moran_process.pipeline.process_lab import submit_post_batch_jobs

submit_post_batch_jobs(
    batch_dir=str(SIM_DATA / COMBINED_BATCH),
    batch_name=COMBINED_BATCH,
    aggregate_job_id=None,  # nothing to wait for; graph_statistics.csv is already there
)

### Sanity check

Loads the combined batch exactly the way `experiment_analysis.ipynb` does. `n_nodes` being
free of NaN is the check that matters: it means every result row found a `graph_props`
match, so nothing fell through the union.

In [ ]:
import pandas as pd

from moran_process.analysis.analysis_utils import (
    build_graph_statistics,
    load_batch_info,
    resolve_results_source,
)

COMBINED_DIR = SIM_DATA / COMBINED_BATCH

info = load_batch_info(COMBINED_DIR)
print("combined from:", [(p["name"], p["n_graphs"]) for p in info["combined_from"]])
print("raw source:   ", resolve_results_source(COMBINED_DIR))

df = build_graph_statistics(
    resolve_results_source(COMBINED_DIR),
    pd.read_csv(COMBINED_DIR / "graph_props.csv"),
    COMBINED_DIR / "graph_statistics.csv",
    r_filter=[1.1],
)
print("\nrows with no graph_props match:", int(df["n_nodes"].isna().sum()))
df["category"].value_counts()